In [0]:
# ============================================================
#  dim_store  —  SCD Type 2  (SIMPLE PySpark, NO SQL MERGE)
# ============================================================
#  SCD2 = "keep history" — every change creates a NEW row, the
#  old row is closed out (is_active='N', end_effective_ts=now).
#
#  4 simple steps:
#    1. Read latest snapshot from silver (one row per store_id).
#    2. Read current ACTIVE rows from the dim.
#    3. Compare business-column hash to flag rows as:
#         NEW       -> insert
#         CHANGE    -> close out old + insert new
#         NO_CHANGE -> ignore
#    4. Apply: DeltaTable.update() to close old, append to insert new.
# ============================================================
from pyspark.sql import functions as F, Window as W
from delta.tables import DeltaTable

dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")

silver_tbl = f"saleslake_{env}.silver_{env}.cleanedStore"
gold_tbl   = f"saleslake_{env}.gold_{env}.dim_store"

BIZ_COLS = ["store_code", "store_name", "store_type", "address", "city", "state", "region_id", "manager_name", "opening_date", "square_feet", "status"]

# Helper: SHA-256 hash of all business columns (concatenated)
def biz_hash(df, cols):
    return df.withColumn("biz_hash", F.sha2(
        F.concat_ws("||",
            *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in cols]),
        256))

# ---------- 1. Latest snapshot per natural key from silver ----------
src = (spark.table(silver_tbl)
            .withColumn("rn", F.row_number().over(
                W.partitionBy("store_id").orderBy(F.col("ingest_ts").desc())))
            .filter("rn = 1").drop("rn", "ingest_ts"))
src = biz_hash(src, BIZ_COLS)

# ---------- 2. Active rows from current dim ----------
dim_delta = DeltaTable.forName(spark, gold_tbl)
dim_active = biz_hash(
    dim_delta.toDF().filter("is_active = 'Y'"),
    BIZ_COLS,
).select("store_id", F.col("biz_hash").alias("dim_hash"))

# ---------- 3. Flag NEW / CHANGE / NO_CHANGE ----------
joined = (src.join(dim_active, on="store_id", how="left")
              .withColumn("rec_flag",
                  F.when(F.col("dim_hash").isNull(),         F.lit("NEW"))
                   .when(F.col("biz_hash") != F.col("dim_hash"), F.lit("CHANGE"))
                   .otherwise(F.lit("NO_CHANGE"))))

# ---------- 4a. Close out CHANGED active rows ----------
changed = [r["store_id"] for r in joined.filter("rec_flag = 'CHANGE'")
                                          .select("store_id").collect()]
if changed:
    dim_delta.update(
        condition = (F.col("store_id").isin(changed)) & (F.col("is_active") == "Y"),
        set = {
            "is_active":        F.lit("N"),
            "end_effective_ts": F.current_timestamp(),
        }
    )
    print(f"Closed out {len(changed)} CHANGED rows")

# ---------- 4b. Append NEW + CHANGED as new active rows ----------
to_insert = (joined.filter(F.col("rec_flag").isin("NEW","CHANGE"))
                   .select("store_id", *BIZ_COLS)
                   .withColumn("is_active",          F.lit("Y"))
                   .withColumn("rec_version",        F.lit(1))
                   .withColumn("start_effective_ts", F.current_timestamp())
                   .withColumn("end_effective_ts",   F.to_timestamp(F.lit("9999-12-31"))))

n_insert = to_insert.count()
if n_insert > 0:
    to_insert.write.format("delta").mode("append").saveAsTable(gold_tbl)
    print(f"Inserted {n_insert} new active rows")

print("SCD2 load complete")
spark.sql(f"""SELECT COUNT(*) AS total,
                     SUM(CASE WHEN is_active='Y' THEN 1 ELSE 0 END) AS active
              FROM {gold_tbl}""").display()